In [ ]:
%load_ext autoreload
%autoreload 2

from funkcje.funkcje import *
from funkcje.wskazniki_kredytowe import oblicz_wskazniki
from funkcje.do_pipeline import *

In [2]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

In [3]:
#Dane
df = pd.read_csv("zbior_1.csv")
#usuwam wybrane zmienne kategoryczne na podstawie tego co w eksploracja_danych.ipynb
X=df.iloc[:, :-1].drop(columns=["szczegolnaFormaPrawna_Symbol", "schemat_wsk_rzis","schemat_wsk_bilans"])
X=oblicz_wskazniki(X)
y=df.iloc[:, -1]

#Podzil na zbior treningowy (70%) i pozostale
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=2137,
    stratify=y
)

#Podzil na zbior walidacyjny (15%) i testowy (15%)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=2137,
    stratify=y_temp
)

# Finalny model regresji

In [4]:

regresja_logistyczna_pipeline = Pipeline([
    ("przygotowanie_cech", PrzygotowanieCech([])),
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        l1_ratio=1, solver="liblinear", C=100, random_state=2137, max_iter=1000
    )),
])

In [21]:
params={'classifier__C': 1,
  'przygotowanie_cech__prog_auc': 0.5,
  'przygotowanie_cech__prog_nan': 0.5,
  'przygotowanie_cech__prog_zer': 0.4,
  'przygotowanie_cech__prog_korelacji': 0.7,
  'przygotowanie_cech__tylko_wskazniki': False}
regresja_logistyczna = clone(regresja_logistyczna_pipeline)
regresja_logistyczna.set_params(**params)
regresja_logistyczna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('przygotowanie_cech', ...), ('imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,kolumny_do_pominiecia,[]
,prog_korelacji,0.7
,prog_auc,0.5
,prog_zer,0.4
,tylko_wskazniki,False


In [22]:
betas = regresja_logistyczna.named_steps["classifier"].coef_[0]

In [23]:
#liczb dodatnich bet
np.sum(betas != 0)

np.int64(55)

In [26]:
print(roc_auc_score(y_valid, regresja_logistyczna.predict_proba(X_valid)[:, 1]))


0.8232317073170732


# Finalny model black-box

In [9]:
xgboost_pipeline=Pipeline([
    ("przygotowanie_cech", PrzygotowanieCech_xgb()),
    ("classifier", XGBClassifier(
        enable_categorical=True,
        tree_method="hist",
        random_state=2137)
    )
])
prams_xgb={'classifier__colsample_bytree': 1.0, 'classifier__gamma': 1, 'classifier__learning_rate': 0.03, 'classifier__max_depth': 2, 'classifier__min_child_weight': 3, 'classifier__n_estimators': 400, 'classifier__reg_alpha': 1, 'classifier__reg_lambda': 10, 'classifier__subsample': 1.0, 'przygotowanie_cech__prog_korelacji': 0.7, 'przygotowanie_cech__prog_nan': 0.25, 'przygotowanie_cech__prog_zer': 0.3, 'przygotowanie_cech__uwzglednij_kategoryczne': False}

In [10]:
black_box = clone(xgboost_pipeline)
black_box.set_params(**prams_xgb)
black_box.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('przygotowanie_cech', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,prog_korelacji,0.7
,prog_nan,0.25
,prog_zer,0.3
,uwzglednij_kategoryczne,False
,kolumny_do_pominiecia,None


In [17]:
black_box.predict_proba(X_valid)[:, 1]
print(roc_auc_score(y_valid, black_box.predict_proba(X_valid)[:, 1]))

0.8396951219512194
